In [ ]:
import os
import sys
import re
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score
import xgboost as xgb

usuario_github = 'federicosolerup'
repositorio_github = 'barra_ventaja_tenis'
url_repositorio = f'https://github.com/{usuario_github}/{repositorio_github}.git'

if not os.path.exists('modelo_xgboost_3tiros.json'):
    if not os.path.exists(repositorio_github):
        os.system(f'git clone {url_repositorio}')
    if os.path.exists(repositorio_github):
        os.chdir(repositorio_github)

ruta_modelo = 'modelo_xgboost_3tiros.json'

if True:  # El dataset se carga incluso cuando existe el modelo entrenado.
    ruta_datos = 'dataset' if os.path.exists('dataset/charting-m-matches.csv') else '.'
    if not os.path.exists(f'{ruta_datos}/charting-m-matches.csv'):
        for posible in ['dataset', '.', f'{repositorio_github}/dataset', '../dataset']:
            if os.path.exists(f'{posible}/charting-m-matches.csv'):
                ruta_datos = posible
                break
    if not os.path.exists(f'{ruta_datos}/charting-m-points-modern.csv') and os.path.exists(f'{ruta_datos}/charting-m-points-modern.csv.zip'):
        with zipfile.ZipFile(f'{ruta_datos}/charting-m-points-modern.csv.zip', 'r') as archivo_zip:
            archivo_zip.extractall(ruta_datos)
    partidos = pd.read_csv(f'{ruta_datos}/charting-m-matches.csv', low_memory=False)
    puntos = pd.read_csv(f'{ruta_datos}/charting-m-points-modern.csv', low_memory=False)
    display(partidos.head())
    display(puntos.head())


In [ ]:
if True:  # Reconstruir el mismo conjunto de prueba, sin reentrenar XGBoost.
    for columna in ['Pl 1 hand', 'Pl 2 hand']:
        partidos[columna] = partidos[columna].astype(str).str.strip().str.upper()

    partidos_diestros = set(partidos[(partidos['Pl 1 hand'] == 'R') & (partidos['Pl 2 hand'] == 'R')]['match_id'])

    puntos_filtrados = puntos[
        puntos['match_id'].isin(partidos_diestros) &
        puntos['PtWinner'].isin(['1', '2', 1, 2])
    ].copy()

    puntos_filtrados['PtWinner'] = puntos_filtrados['PtWinner'].astype(int)
    puntos_filtrados['Svr'] = puntos_filtrados['Svr'].astype(int)
    puntos_filtrados['secuencia_peloteo'] = np.where(
        puntos_filtrados['2nd'].notna(), puntos_filtrados['2nd'], puntos_filtrados['1st']
    )
    puntos_filtrados = puntos_filtrados.dropna(subset=['secuencia_peloteo']).copy()

    print(f'Puntos totales cargados: {len(puntos):,}')
    print(f'Puntos filtrados (ATP diestros): {len(puntos_filtrados):,}')

In [ ]:
mapeo_golpes = {
    'f': 'derecha', 'b': 'reves', 's': 'slice_reves', 'r': 'slice_derecha',
    'v': 'volea', 'z': 'media_volea', 'o': 'smash',
    '4': 'saque_abierto', '5': 'saque_cuerpo', '6': 'saque_t'
}

mapeo_direccion = {
    '0': 'indefinida', '1': 'paralelo', '2': 'centro', '3': 'cruzado'
}

def procesar_peloteo(peloteo):
    tiros = [t for t in re.split(r'(?=[fbvsroz])', str(peloteo)) if t.strip()] if isinstance(peloteo, str) else peloteo
    analizados = []
    for t in tiros:
        s = t.lower()
        if s.startswith('c') and len(s) > 1 and s[1] in '456':
            s = s[1:]
        tipo = mapeo_golpes.get(s[0], 'ninguno')
        busqueda_dir = re.search(r'^[fbvsroz]\+?([123])', t, re.IGNORECASE)
        valor_dir = busqueda_dir.group(1) if busqueda_dir else '0'
        direccion = mapeo_direccion.get(valor_dir, 'indefinida')
        busqueda_prof = re.search(r'^[fbvsroz]\+?[123]?([789])', t, re.IGNORECASE)
        valor_prof = busqueda_prof.group(1) if busqueda_prof else '0'
        analizados.append({
            'golpe': tipo,
            'direccion': direccion,
            'profundidad_corta': int(valor_prof == '7'),
            'profundidad_media': int(valor_prof == '8'),
            'profundidad_profunda': int(valor_prof == '9'),
            'subida_a_red': int('+' in t)
        })

    filas = []
    vacio = {
        'golpe': 'ninguno', 'direccion': 'indefinida',
        'profundidad_corta': 0, 'profundidad_media': 0,
        'profundidad_profunda': 0, 'subida_a_red': 0
    }
    for i, tiro_actual in enumerate(analizados):
        previo = analizados[i - 1] if i >= 1 else vacio
        anteprevio = analizados[i - 2] if i >= 2 else vacio
        filas.append({
            'sacador_al_tiro': int(i % 2 == 0),
            **tiro_actual,
            **{f'prev_{k}': v for k, v in previo.items()},
            **{f'anteprev_{k}': v for k, v in anteprevio.items()}
        })
    return filas

if True:  # Reconstruir el mismo conjunto de prueba, sin reentrenar XGBoost.
    tiros_totales = []
    for id_partido, sacador, ganador, peloteo in zip(
        puntos_filtrados['match_id'], puntos_filtrados['Svr'], puntos_filtrados['PtWinner'], puntos_filtrados['secuencia_peloteo']
    ):
        for fila in procesar_peloteo(peloteo):
            fila['match_id'] = str(id_partido)
            fila['target'] = int(ganador == sacador)
            tiros_totales.append(fila)

    dataset_tiros = pd.DataFrame(tiros_totales)
    matriz_x = pd.get_dummies(dataset_tiros.drop(columns=['match_id', 'target']))
    vector_y = dataset_tiros['target']
    partidos_grupos = dataset_tiros['match_id']

    print(f'Tiros procesados: {len(dataset_tiros):,}')
    print(f'Columnas de caracteristicas: {matriz_x.shape[1]}')
    display(matriz_x.head())

In [ ]:
if True:  # Reconstruir el mismo conjunto de prueba, sin reentrenar XGBoost.
    separador = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
    indices_entrenamiento, indices_prueba = next(separador.split(matriz_x, vector_y, partidos_grupos))

    x_entrenamiento = matriz_x.iloc[indices_entrenamiento]
    x_prueba = matriz_x.iloc[indices_prueba]
    y_entrenamiento = vector_y.iloc[indices_entrenamiento]
    y_prueba = vector_y.iloc[indices_prueba]

    partidos_entrenamiento = set(partidos_grupos.iloc[indices_entrenamiento])
    partidos_prueba = set(partidos_grupos.iloc[indices_prueba])

    columnas_t = [c for c in matriz_x.columns if not c.startswith('prev_') and not c.startswith('anteprev_')]
    columnas_t_prev = [c for c in matriz_x.columns if not c.startswith('anteprev_')]
    columnas_todas = list(matriz_x.columns)

    print(f'Partidos en entrenamiento: {len(partidos_entrenamiento):,} | Tiros: {len(x_entrenamiento):,}')
    print(f'Partidos en prueba: {len(partidos_prueba):,} | Tiros: {len(x_prueba):,}')
    print(f'Variables t: {len(columnas_t)}')
    print(f'Variables t, t-1: {len(columnas_t_prev)}')
    print(f'Variables t, t-1, t-2: {len(columnas_todas)}')

In [ ]:
modelo_completo = xgb.XGBClassifier()

if True:  # Evaluar siempre; cargar modelo guardado en lugar de reentrenarlo cuando existe.
    def calcular_ece(valores_reales, probabilidades, num_intervalos=10):
        intervalos = np.linspace(0, 1, num_intervalos + 1)
        error_calibracion = 0.0
        total = len(valores_reales)
        reales_array = np.asarray(valores_reales)
        for i in range(num_intervalos):
            if i == 0:
                mascara = (probabilidades >= intervalos[i]) & (probabilidades <= intervalos[i + 1])
            else:
                mascara = (probabilidades > intervalos[i]) & (probabilidades <= intervalos[i + 1])
            if np.sum(mascara) > 0:
                error_calibracion += (np.sum(mascara) / total) * np.abs(np.mean(probabilidades[mascara]) - np.mean(reales_array[mascara]))
        return error_calibracion

    def evaluar_modelo(nombre, probabilidades, predicciones):
        exactitud = accuracy_score(y_prueba, predicciones)
        auc = roc_auc_score(y_prueba, probabilidades)
        brier = brier_score_loss(y_prueba, probabilidades)
        ece = calcular_ece(y_prueba, probabilidades)
        p_sac = precision_score(y_prueba, predicciones, pos_label=1)
        r_sac = recall_score(y_prueba, predicciones, pos_label=1)
        f_sac = f1_score(y_prueba, predicciones, pos_label=1)
        p_ret = precision_score(y_prueba, predicciones, pos_label=0, zero_division=0)
        r_ret = recall_score(y_prueba, predicciones, pos_label=0, zero_division=0)
        f_ret = f1_score(y_prueba, predicciones, pos_label=0, zero_division=0)
        return {
            'Modelo': nombre,
            'Accuracy': f'{exactitud * 100:.2f}%',
            'AUC-ROC': f'{auc * 100:.2f}%',
            'Brier Score': f'{brier:.4f}',
            'ECE': f'{ece * 100:.2f}%',
            'Prec. Sac.': f'{p_sac * 100:.2f}%',
            'Rec. Sac.': f'{r_sac * 100:.2f}%',
            'F1 Sac.': f'{f_sac * 100:.2f}%',
            'Prec. Rest.': f'{p_ret * 100:.2f}%',
            'Rec. Rest.': f'{r_ret * 100:.2f}%',
            'F1 Rest.': f'{f_ret * 100:.2f}%'
        }

    parametros_xgb = dict(
        max_depth=8,
        learning_rate=0.05,
        n_estimators=200,
        subsample=0.9,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        tree_method='hist',
        n_jobs=-1
    )

    filas_ablacion = []

    # Baseline aleatorio reproducible, independiente de las etiquetas reales.
    # Cada predicción se obtiene con una moneda equilibrada.
    rng_dummy = np.random.default_rng(42)
    pred_dummy_50 = rng_dummy.integers(0, 2, size=len(y_prueba))
    prob_dummy_50 = np.full(len(y_prueba), 0.50)
    res_dummy_50 = evaluar_modelo('Dummy Uniforme (aleatorio)', prob_dummy_50, pred_dummy_50)
    filas_ablacion.append(res_dummy_50)

    p_prevalencia = float(np.mean(y_entrenamiento))
    prob_dummy_mayoria = np.full(len(y_prueba), p_prevalencia)
    pred_dummy_mayoria = np.ones(len(y_prueba), dtype=int)
    res_dummy_mayoria = evaluar_modelo('Dummy (p = 0.567)', prob_dummy_mayoria, pred_dummy_mayoria)
    filas_ablacion.append(res_dummy_mayoria)

    if os.path.exists(ruta_modelo):
        modelo_completo.load_model(ruta_modelo)
        print(f'Modelo XGBoost cargado desde {ruta_modelo}; no se reentrena.')
        columnas_modelo = modelo_completo.get_booster().feature_names
        if columnas_modelo is None:
            raise ValueError('El modelo guardado no contiene los nombres de las variables.')
        faltantes = set(columnas_modelo) - set(x_prueba.columns)
        if faltantes:
            raise ValueError(f'Faltan columnas requeridas por el modelo: {sorted(faltantes)}')
        prob_completo = modelo_completo.predict_proba(x_prueba.reindex(columns=columnas_modelo))[:, 1]
        pred_completo = modelo_completo.predict(x_prueba.reindex(columns=columnas_modelo))
        filas_ablacion.append(evaluar_modelo('XGBoost (t, t-1, t-2)', prob_completo, pred_completo))
        print('Los modelos de ablación t y t,t-1 no están guardados; no se vuelven a entrenar.')
    else:
        modelo_t = xgb.XGBClassifier(**parametros_xgb)
        modelo_t.fit(x_entrenamiento[columnas_t], y_entrenamiento)
        prob_t = modelo_t.predict_proba(x_prueba[columnas_t])[:, 1]
        pred_t = modelo_t.predict(x_prueba[columnas_t])
        filas_ablacion.append(evaluar_modelo('XGBoost (t)', prob_t, pred_t))

        modelo_t_prev = xgb.XGBClassifier(**parametros_xgb)
        modelo_t_prev.fit(x_entrenamiento[columnas_t_prev], y_entrenamiento)
        prob_t_prev = modelo_t_prev.predict_proba(x_prueba[columnas_t_prev])[:, 1]
        pred_t_prev = modelo_t_prev.predict(x_prueba[columnas_t_prev])
        filas_ablacion.append(evaluar_modelo('XGBoost (t, t-1)', prob_t_prev, pred_t_prev))

        modelo_completo = xgb.XGBClassifier(**parametros_xgb)
        modelo_completo.fit(x_entrenamiento[columnas_todas], y_entrenamiento)
        prob_completo = modelo_completo.predict_proba(x_prueba[columnas_todas])[:, 1]
        pred_completo = modelo_completo.predict(x_prueba[columnas_todas])
        filas_ablacion.append(evaluar_modelo('XGBoost (t, t-1, t-2)', prob_completo, pred_completo))

        modelo_completo.save_model(ruta_modelo)
    display(pd.DataFrame(filas_ablacion))


In [ ]:
# Auditoría reproducible del modelo XGBoost guardado y de los dos Dummy.
# Esta celda NO entrena modelos ni modifica el archivo JSON.
from sklearn.metrics import confusion_matrix, log_loss
import hashlib

if 'prob_completo' not in globals() or 'pred_completo' not in globals():
    raise RuntimeError('Ejecutá primero las celdas anteriores para reconstruir el test y cargar XGBoost.')

reales = np.asarray(y_prueba).astype(int)
probabilidades = np.asarray(prob_completo, dtype=float)
predicciones = np.asarray(pred_completo).astype(int)
matriz = confusion_matrix(reales, predicciones, labels=[0, 1])
tn, fp, fn, tp = matriz.ravel()

print('MODELO Y CONJUNTO DE PRUEBA')
print('Ruta del modelo:', os.path.abspath(ruta_modelo))
with open(ruta_modelo, 'rb') as archivo:
    print('SHA-256 del JSON:', hashlib.file_digest(archivo, 'sha256').hexdigest())
print('Partidos de prueba:', len(partidos_prueba))
print('Tiros de prueba:', len(reales))
print('Sacador (clase 1):', int(reales.sum()))
print('Restador (clase 0):', int((reales == 0).sum()))
print('Características del modelo:', len(modelo_completo.get_booster().feature_names))

print('\nMATRIZ DE CONFUSIÓN XGBOOST (filas = clase real, columnas = predicción)')
display(pd.DataFrame(matriz, index=['Real restador (0)', 'Real sacador (1)'],
                     columns=['Pred. restador (0)', 'Pred. sacador (1)']))
print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')

print('\nMÉTRICAS SIN REDONDEAR')
for nombre, probabilidades_modelo, predicciones_modelo in [
    ('Dummy Uniforme', prob_dummy_50, pred_dummy_50),
    ('Dummy Mayoría', prob_dummy_mayoria, pred_dummy_mayoria),
    ('XGBoost tres tiros', probabilidades, predicciones),
]:
    print('\n' + nombre)
    print('Accuracy:', format(accuracy_score(reales, predicciones_modelo), '.10f'))
    print('AUC-ROC:', format(roc_auc_score(reales, probabilidades_modelo), '.10f'))
    print('Brier Score:', format(brier_score_loss(reales, probabilidades_modelo), '.10f'))
    print('Log Loss:', format(log_loss(reales, probabilidades_modelo, labels=[0, 1]), '.10f'))
    print('ECE:', format(calcular_ece(reales, probabilidades_modelo), '.10f'))
    for etiqueta, clase in [('Sacador', 1), ('Restador', 0)]:
        print(etiqueta, 'Precision:', format(precision_score(reales, predicciones_modelo, pos_label=clase, zero_division=0), '.10f'),
              'Recall:', format(recall_score(reales, predicciones_modelo, pos_label=clase, zero_division=0), '.10f'),
              'F1:', format(f1_score(reales, predicciones_modelo, pos_label=clase, zero_division=0), '.10f'))

print('\nCONTROL DE UMBRAL')
pred_umbral = (probabilidades >= 0.5).astype(int)
print('Diferencias entre predict() y umbral 0.5:', int(np.sum(predicciones != pred_umbral)))


In [ ]:
tiros_estudio = ['6', 's29', 'f27', 'b19', 'f27', 'f+38', 'f18', 'z37*']
etiquetas = [
    '0. Estado previo al saque',
    '1. 1er Saque falta en red (DP)',
    '2. 2do Saque a la T (DP)',
    '3. Slice de revés al centro profundo (F)',
    '4. Derecha al centro corta (DP)',
    '5. Revés paralelo profundo (F)',
    '6. Derecha al centro corta (DP)',
    '7. Approach derecha cruzada media (F)',
    '8. Derecha paralela media (DP)',
    '9. Volea de revés cruzada corta ganadora (F)'
]
ejecutores = ['Del Potro', 'Del Potro', 'Del Potro', 'Federer', 'Del Potro', 'Federer', 'Del Potro', 'Federer', 'Del Potro', 'Federer']

columnas_modelo = modelo_completo.get_booster().feature_names
df_simulacion = pd.get_dummies(pd.DataFrame(procesar_peloteo(tiros_estudio[:-1]))).reindex(columns=columnas_modelo, fill_value=0)
probabilidades_del_potro = [0.64, 0.51] + [round(float(p), 2) for p in modelo_completo.predict_proba(df_simulacion)[:, 1]] + [0.00]
probabilidades_federer = [round(1.0 - p, 2) for p in probabilidades_del_potro]

tabla_caso_estudio = pd.DataFrame({
    'Paso': range(len(etiquetas)),
    'Descripción del Tiro': etiquetas,
    'Ejecutor': ejecutores,
    'P(Del Potro)': [f'{p:.2f}' for p in probabilidades_del_potro],
    'P(Federer)': [f'{p:.2f}' for p in probabilidades_federer],
    'ΔP(Sacador)': ['0.00'] + [f'{p2 - p1:+.2f}' for p1, p2 in zip(probabilidades_del_potro[:-1], probabilidades_del_potro[1:])]
})
display(tabla_caso_estudio)

ruta_grafico = 'eval_bar_simulation.png'

posiciones = np.arange(len(probabilidades_del_potro))
figura, eje = plt.subplots(figsize=(12.5, 6.2), dpi=300)

for i, ej in enumerate(ejecutores):
    eje.axvspan(i - 0.5, i + 0.5, facecolor='#fff1f2' if ej == 'Del Potro' else '#f0f9ff', alpha=0.85, zorder=1)

eje.axhline(0.50, color='#94a3b8', linestyle='--', linewidth=1.5, label='Neutralidad táctica (0.50)', zorder=2)
eje.plot(posiciones, probabilidades_federer, marker='o', markersize=9, linewidth=3.0, color='#0284c7', label='P(Victoria Federer - Restador)', zorder=4)
eje.plot(posiciones, probabilidades_del_potro, marker='s', markersize=8, linewidth=2.2, linestyle=':', color='#e11d48', label='P(Victoria Del Potro - Sacador)', zorder=4)

caja_azul = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#bae6fd', linewidth=0.8, alpha=0.92)
caja_roja = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='#fecdd3', linewidth=0.8, alpha=0.92)

for i, (p_f, p_d) in enumerate(zip(probabilidades_federer, probabilidades_del_potro)):
    (p_sup, c_sup, b_sup), (p_inf, c_inf, b_inf) = (
        ((p_d, '#be123c', caja_roja), (p_f, '#0369a1', caja_azul)) if p_d >= p_f
        else ((p_f, '#0369a1', caja_azul), (p_d, '#be123c', caja_roja))
    )
    eje.text(i - 0.16 if p_sup == 1.0 else i, p_sup if p_sup == 1.0 else p_sup + 0.055, f'{p_sup*100:.0f}%',
            ha='right' if p_sup == 1.0 else 'center', va='center' if p_sup == 1.0 else 'bottom',
            fontsize=10.5 if p_sup == 1.0 else 10, fontweight='bold', color=c_sup, bbox=b_sup, zorder=6)
    eje.text(i, p_inf + 0.05 if p_inf == 0.0 else p_inf - 0.065, f'{p_inf*100:.0f}%',
            ha='center', va='bottom' if p_inf == 0.0 else 'top',
            fontsize=10, fontweight='bold', color=c_inf, bbox=b_inf, zorder=6)

eje.set_ylim(-0.08, 1.16)
eje.set_xlim(-0.6, len(posiciones) - 0.4)
eje.set_xticks(posiciones)
eje.set_xticklabels(etiquetas, fontsize=9.5, fontweight='medium', rotation=26, ha='right')
eje.set_ylabel('Probabilidad Condicional de Victoria', fontsize=12, fontweight='bold', labelpad=10)
eje.set_title('Evolución de la Probabilidad Condicional de Victoria - Caso de estudio', fontsize=12, fontweight='bold', pad=15)
eje.grid(True, linestyle=':', alpha=0.55, color='#94a3b8', zorder=2)
eje.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#cbd5e1', framealpha=0.95, fontsize=10.5)

plt.tight_layout()
plt.savefig(ruta_grafico, dpi=300)
plt.show()